# Silver layer: table definitions

This notebook creates and inspects the empty Sprint 1 Silver schema from `../sql/schema/silver.sql`.

**Run only against a disposable local database.** The reset cell drops the seven Silver tables from `../data/lakehouse.duckdb` before rebuilding the schema. Use `scripts/run_sprint1.py` to run the sample without modifying that database.

## 0. Setup

In [5]:
import duckdb

con = duckdb.connect('../data/lakehouse.duckdb')
print('DuckDB version:', duckdb.__version__)

DuckDB version: 2.0.0


In [6]:
con.sql("""
        DROP TABLE IF EXISTS silver_fact_trips;
        DROP TABLE IF EXISTS silver_fact_overnight;
        DROP TABLE IF EXISTS silver_zone_metrics;
        DROP TABLE IF EXISTS silver_holidays;
        DROP TABLE IF EXISTS silver_dim_zone;
        DROP TABLE IF EXISTS silver_dim_province;
        DROP TABLE IF EXISTS silver_dim_ccaa;
    """)

SuccessBOOLEAN


## 1. Rules for all the Silver tables

- Names in English and real types (DATE, TIMESTAMP, DOUBLE, INTEGER, BOOLEAN, GEOMETRY).
- Empty values and INE placeholders like '.' become NULL.
- INE data only for 2023.
- A zone is always identified by two columns: zone_level ('distritos', 'municipios' or 'GAU') and zone_id.

Two columns are still not decided and are not included: municipality_id and gau_id in silver_dim_zone, and year in silver_zone_metrics.

## 2. Dimensions

silver_dim_ccaa: one row per region. Comes from bronze_ine_provinces.

silver_dim_province: one row per province, with the code of its region. Comes from bronze_ine_provinces. Relation: ccaa_code points to silver_dim_ccaa.

In [7]:
from pathlib import Path

con.execute(Path("../sql/schema/silver.sql").read_text(encoding="utf-8"))

CountBIGINT


silver_dim_zone: one row per zone and level. Comes from bronze_mitma_zone_names, bronze_mitma_zone_geometries and bronze_mitma_zone_mapping (for the province). The province is empty for foreign zones and for the 49 unpopulated lands. Relation: province_code points to silver_dim_province.

silver_holidays: one row per holiday and region. Comes from bronze_boe_holidays. The national holidays are the days that appear in the 19 regions. Relation: ccaa_code points to silver_dim_ccaa.

## 3. Zone metrics

silver_zone_metrics: population, income and businesses per zone. Comes from bronze_ine_adrh and bronze_ine_dirce, moved to MITMA zones with bronze_mitma_zone_mapping. The business columns are empty for districts, because DIRCE is only published per municipality. Relation: (zone_level, zone_id) points to silver_dim_zone.

## 4. Facts

silver_fact_trips: trips per hour, origin, destination and activities, at each level. Comes from bronze_mitma_od_daily. Its key is all the columns except trips. Relation: (zone_level, origin_zone_id) and (zone_level, destination_zone_id) point to silver_dim_zone.

silver_fact_overnight: people per day, residence district and zone where they slept. Comes from bronze_mitma_overnight_daily. The residence is always a district, also when zone_level is 'municipios' or 'GAU'. Relation: (zone_level, overnight_zone_id) points to silver_dim_zone. ('distritos', residence_district_id) also points to silver_dim_zone, but it can't be a foreign key (see the rules above).

## 5. Result

The Silver tables and their columns. All of them are empty.

In [12]:
con.sql("""
        SELECT table_name, column_name, data_type, is_nullable
        FROM information_schema.columns
        WHERE table_name LIKE 'silver%'
        ORDER BY table_name, ordinal_position
    """).show(max_rows=100)

┌───────────────────────┬────────────────────────────────────────┬───────────┬─────────────┐
│      table_name       │              column_name               │ data_type │ is_nullable │
│        varchar        │                varchar                 │  varchar  │   varchar   │
├───────────────────────┼────────────────────────────────────────┼───────────┼─────────────┤
│ silver_dim_ccaa       │ ccaa_code                              │ VARCHAR   │ NO          │
│ silver_dim_ccaa       │ ccaa                                   │ VARCHAR   │ NO          │
│ silver_dim_province   │ province_code                          │ VARCHAR   │ NO          │
│ silver_dim_province   │ province_name                          │ VARCHAR   │ NO          │
│ silver_dim_province   │ ccaa_code                              │ VARCHAR   │ NO          │
│ silver_dim_zone       │ zone_level                             │ VARCHAR   │ NO          │
│ silver_dim_zone       │ zone_id                                │ VAR

And the keys of each table.

In [13]:
con.sql("""
        SELECT table_name, constraint_type, constraint_column_names
        FROM duckdb_constraints()
        WHERE table_name LIKE 'silver%' AND constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY')
        ORDER BY table_name, constraint_type DESC
    """).show(max_rows=100)

┌───────────────────────┬─────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│      table_name       │ constraint_type │                                         constraint_column_names                                         │
│        varchar        │     varchar     │                                                varchar[]                                                │
├───────────────────────┼─────────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ silver_dim_ccaa       │ PRIMARY KEY     │ [ccaa_code]                                                                                             │
│ silver_dim_province   │ PRIMARY KEY     │ [province_code]                                                                                         │
│ silver_dim_province   │ FOREIGN KEY     │ [ccaa_code]                                             

In [14]:
con.close()